# VaR Demo Data Runner

This notebook provides a thin, standard wrapper around the generator script so you can rerun and validate fixtures quickly.

In [4]:
from pathlib import Path

# Parameters
output_dir = "data/var_demo"
rerun_generator = True

cwd = Path.cwd()
if (cwd / "src" / "simulated_data_lab" / "generate_var_demo_data.py").exists():
    project_root = cwd
elif cwd.name == "notebooks" and (cwd.parent / "src" / "simulated_data_lab" / "generate_var_demo_data.py").exists():
    project_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from simulated_data_lab or simulated_data_lab/notebooks.")

print("project_root:", project_root)
print("output_dir:", output_dir)

project_root: /Users/kamlesh/dev/git/Demos/financial_quant/simulated_data_lab
output_dir: data/var_demo


In [5]:
import shutil
import subprocess

if rerun_generator:
    script = "src/simulated_data_lab/generate_var_demo_data.py"
    venv_python = project_root / ".venv" / "bin" / "python"

    if venv_python.exists():
        cmd = [str(venv_python), script, "--output-dir", output_dir]
    else:
        uv_bin = shutil.which("uv")
        if uv_bin is None:
            raise RuntimeError(
                "Could not find project .venv Python or uv executable. "
                "Run `uv sync` in simulated_data_lab, or select the project interpreter in VS Code."
            )
        cmd = [uv_bin, "run", "python", script, "--output-dir", output_dir]

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=project_root, check=True)
else:
    print("Skipping generator run (rerun_generator=False).")

Running: /Users/kamlesh/dev/git/Demos/financial_quant/simulated_data_lab/.venv/bin/python src/simulated_data_lab/generate_var_demo_data.py --output-dir data/var_demo
Generated VaR demo fixtures:
  portfolio rows: 500
  historical_returns rows: 250,000
  market_params rows: 500
  sector_correlation shape: (11, 11)
Files:
  portfolio.parquet: 19 KB
  historical_returns.parquet: 1818 KB
  market_params.parquet: 21 KB
  sector_correlation.parquet: 9 KB


In [6]:
import pandas as pd

base = project_root / output_dir
port = pd.read_parquet(base / "portfolio.parquet")
rets = pd.read_parquet(base / "historical_returns.parquet")
params = pd.read_parquet(base / "market_params.parquet")
scorr = pd.read_parquet(base / "sector_correlation.parquet")

assert len(port) == 500, f"portfolio: {len(port)}"
assert len(rets) == 250_000, f"returns: {len(rets)}"
assert len(params) == 500, f"params: {len(params)}"
assert scorr.shape == (11, 11), f"sector_corr shape: {scorr.shape}"
assert set(port.ticker) == set(rets.ticker.unique()), "ticker mismatch port/returns"
assert set(port.ticker) == set(params.ticker), "ticker mismatch port/params"
assert port.desk.nunique() == 5
assert port.groupby("desk")["book"].nunique().eq(3).all()
assert rets.date.nunique() == 500

print("All assertions passed.")
print("portfolio rows:", len(port))
print("historical_returns rows:", len(rets))
print("market_params rows:", len(params))
print("sector_correlation shape:", scorr.shape)

All assertions passed.
portfolio rows: 500
historical_returns rows: 250000
market_params rows: 500
sector_correlation shape: (11, 11)


In [7]:
print("Desk counts:")
display(port["desk"].value_counts().sort_index().to_frame("positions"))

short_ratio = (port["quantity"] < 0).mean()
print(f"Short ratio: {short_ratio:.2%}")

print("\nReturn quantiles:")
display(rets["return"].quantile([0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]).to_frame("return"))

print("\nDaily vol summary:")
display(params["daily_vol"].describe().to_frame("daily_vol"))

print("\nSector correlation matrix:")
display(scorr)

Desk counts:


,positions
desk,
AsiaPac_Equities,100
EmergingMarkets,100
European_Equities,100
US_LargeCap,100
US_SmallMidCap,100


Short ratio: 19.60%

Return quantiles:


,return
0.001,-0.066087
0.010,-0.041584
0.050,-0.025046
0.500,0.000397
0.950,0.026825
0.990,0.044291
0.999,0.072219



Daily vol summary:


,daily_vol
count,500.000000
mean,0.015793
std,0.003603
min,0.009215
25%,0.012663
50%,0.015780
75%,0.018857
max,0.022915



Sector correlation matrix:


sector,Technology,Financials,Healthcare,ConsumerDiscretionary,ConsumerStaples,Energy,Materials,Industrials,Utilities,RealEstate,CommunicationServices
sector,,,,,,,,,,,
Technology,1.000000,0.437713,0.483958,0.432224,0.431038,0.488917,0.382348,0.418768,0.341328,0.369123,0.394960
Financials,0.437713,1.000000,0.388921,0.464146,0.377793,0.490733,0.385858,0.418179,0.346629,0.402555,0.372134
Healthcare,0.483958,0.388921,1.000000,0.409969,0.398826,0.423491,0.384614,0.407955,0.391071,0.329433,0.396786
ConsumerDiscretionary,0.432224,0.464146,0.409969,1.000000,0.391434,0.485497,0.343639,0.392936,0.412449,0.466832,0.425384
ConsumerStaples,0.431038,0.377793,0.398826,0.391434,1.000000,0.479532,0.348849,0.449147,0.317950,0.414301,0.449269
Energy,0.488917,0.490733,0.423491,0.485497,0.479532,1.000000,0.407726,0.472648,0.348278,0.445451,0.436060
Materials,0.382348,0.385858,0.384614,0.343639,0.348849,0.407726,1.000000,0.394368,0.315732,0.319333,0.341927
Industrials,0.418768,0.418179,0.407955,0.392936,0.449147,0.472648,0.394368,1.000000,0.389297,0.415425,0.419451
Utilities,0.341328,0.346629,0.391071,0.412449,0.317950,0.348278,0.315732,0.389297,1.000000,0.319641,0.383031
